# Laboratorio SSEE — La degeneración $w_0$–$w_a$–$H_0$

**Pregunta (de Mike, 2026-06-14):** el ajuste libre CPL da distintos $w_0,w_a$ según el prior de $H_0$.
¿Es el prior de Planck o la geometría BAO lo que los mueve? Y si usamos el $H_0$ **algebraico** de SSEE,
¿el ajuste libre se acerca a la predicción algebraica $(w_0,w_a)=(-0.840,-0.670)$?

**Idea del método.** No relanzamos el MCMC: tomamos la cadena CPL ya corrida (que usó el prior de Planck)
y la **reponderamos** (*importance sampling*) a otros priors de $H_0$. Cada muestra recibe un peso
$w = \mathcal{N}(H_0;\mu_{\rm nuevo},\sigma)/\mathcal{N}(H_0;\mu_{\rm Planck},\sigma)$.
Válido porque la *verosimilitud* (los datos) no cambia: solo cambiamos el prior.

> ⚠️ **Caveat honesto:** esto cambia SOLO el prior de $H_0$. El test *completo* ("CPL con todo el fondo MIRA/H_alg")
> también remapearía $r_d$ y $\Omega_m$. Para la palabra final hay que **relanzar** el MCMC. Esto es una mirada
> rápida pero fiable (vigilar el ESS efectivo en cada reponderación).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cadena CPL pública: columnas [H0, Om, w0, wa, Obh2]
chain = np.load('../results/logs/mcmc_chains_professional_CPL_ckpt.npz')['chain']
H0, Om, w0, wa, Obh2 = chain.T
print('muestras:', chain.shape[0])

# Constantes de referencia
SSEE = (-0.840, -0.670)          # predicción algebraica (fija, no se ajusta)
H_MIRA, H_PLANCK, H_ALG = 67.037, 67.36, 67.962
SIGMA_H = 0.54                    # ancho del prior de H0 (Planck-error)
print('mediana cruda CPL: w0=%.3f  wa=%.3f  H0=%.2f' % (np.median(w0), np.median(wa), np.median(H0)))

## 1. La degeneración: $w_0$ y $w_a$ no son independientes

La BAO no fija $w_0$ y $w_a$ por separado, solo una **combinación**. Si la correlación es muy negativa,
el posterior es una **elipse alargada** (una casi-línea): puedes mover $w_0$ a cambio de mover $w_a$
en sentido contrario sin empeorar el ajuste.

In [ ]:
r = np.corrcoef(w0, wa)[0, 1]
C = np.cov(w0, wa)
evals, evecs = np.linalg.eigh(C)
slope = evecs[1, 1] / evecs[0, 1]   # pendiente del eje principal (dwa/dw0)
print('correlacion w0-wa = %+.3f' % r)
print('pendiente de la degeneracion dwa/dw0 = %+.2f' % slope)

# scatter (submuestreado) del posterior CPL
idx = np.random.default_rng(0).choice(len(w0), 40000, replace=False)
plt.figure(figsize=(7, 6))
plt.hexbin(w0[idx], wa[idx], gridsize=60, cmap='Blues', mincnt=1)
plt.plot(*SSEE, 'r*', ms=20, label='SSEE algebraico')
plt.xlabel('$w_0$'); plt.ylabel('$w_a$')
plt.title('Posterior CPL (prior Planck) — degeneracion w0-wa  (rho=%.2f)' % r)
plt.legend(); plt.tight_layout(); plt.show()

## 2. Caminar por la línea: barrido del prior de $H_0$

Reponderamos la cadena a una rejilla de centros de prior $H_0$ desde 66 hasta 69 km/s/Mpc.
Para cada uno calculamos la **mediana reponderada** de $(w_0,w_a)$ y el **ESS** (tamaño de muestra
efectivo: si baja mucho, la reponderación deja de ser fiable).

In [ ]:
def reweight_to(mu_new, mu_old=H_PLANCK, s=SIGMA_H):
    """Pesos de importance sampling para cambiar el centro del prior de H0."""
    logw = -0.5 * ((H0 - mu_new) / s) ** 2 + 0.5 * ((H0 - mu_old) / s) ** 2
    w = np.exp(logw - logw.max())
    ess = w.sum() ** 2 / (w ** 2).sum()
    return w, ess

def wmedian(x, w):
    i = np.argsort(x); x, w = x[i], w[i]
    c = np.cumsum(w); c /= c[-1]
    return np.interp(0.5, c, x)

grid = np.linspace(66.0, 69.0, 25)
traj_w0, traj_wa, ess_frac = [], [], []
for mu in grid:
    w, ess = reweight_to(mu)
    traj_w0.append(wmedian(w0, w)); traj_wa.append(wmedian(wa, w))
    ess_frac.append(ess / len(w))
traj_w0, traj_wa, ess_frac = map(np.array, (traj_w0, traj_wa, ess_frac))

# puntos de referencia concretos
for name, mu in [('H_MIRA', H_MIRA), ('Planck', H_PLANCK), ('H_alg', H_ALG)]:
    w, ess = reweight_to(mu)
    print('CPL @ %-7s (H0=%.2f):  w0=%+.3f  wa=%+.3f   ESS=%.0f%%'
          % (name, mu, wmedian(w0, w), wmedian(wa, w), 100 * ess / len(w)))
print('SSEE algebraico        :  w0=%+.3f  wa=%+.3f' % SSEE)

In [ ]:
# Distancia de Mahalanobis SSEE -> mediana CPL reponderada, usando la covarianza del posterior
Cinv = np.linalg.inv(C)
def mahal(p, q):
    d = np.array(p) - np.array(q)
    return np.sqrt(d @ Cinv @ d)

for name, mu in [('H_MIRA', H_MIRA), ('Planck', H_PLANCK), ('H_alg', H_ALG)]:
    w, _ = reweight_to(mu)
    med = (wmedian(w0, w), wmedian(wa, w))
    print('SSEE vs CPL@%-7s : %.2f sigma' % (name, mahal(SSEE, med)))

In [ ]:
# Visualizacion: la trayectoria de la mediana CPL al barrer el prior de H0, hacia SSEE
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))

sc = ax[0].scatter(traj_w0, traj_wa, c=grid, cmap='viridis', s=60, zorder=3)
ax[0].plot(traj_w0, traj_wa, 'k-', alpha=0.3, zorder=2)
ax[0].plot(*SSEE, 'r*', ms=22, label='SSEE algebraico', zorder=4)
for name, mu, col in [('MIRA', H_MIRA, 'tab:blue'), ('Planck', H_PLANCK, 'tab:orange'), ('H_alg', H_ALG, 'tab:green')]:
    w, _ = reweight_to(mu)
    ax[0].plot(wmedian(w0, w), wmedian(wa, w), 'o', color=col, ms=11, mec='k', label='CPL@'+name, zorder=4)
ax[0].set_xlabel('$w_0$'); ax[0].set_ylabel('$w_a$')
ax[0].set_title('Mediana CPL al subir el prior de H0 -> tiende a SSEE')
plt.colorbar(sc, ax=ax[0], label='centro prior $H_0$')
ax[0].legend(fontsize=8)

ax[1].plot(grid, 100 * ess_frac, 'b-o', ms=4)
ax[1].axhline(10, color='r', ls='--', label='10% (piso de fiabilidad)')
ax[1].set_xlabel('centro prior $H_0$'); ax[1].set_ylabel('ESS efectivo [%]')
ax[1].set_title('Fiabilidad de la reponderacion'); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Lectura honesta

**Lo sólido:**
- La correlación $w_0$–$w_a \approx -0.95$: el posterior es casi una línea. La BAO fija una *combinación*, no
  $w_0$ y $w_a$ por separado.
- Al **subir** el prior de $H_0$ (MIRA → Planck → H_alg), la mediana libre de CPL **se desliza por esa línea hacia SSEE**.
- En el $H_0$ algebraico de SSEE (67.96) el ajuste libre cae mucho más cerca de la predicción que con el prior de Planck.

**Lo que NO prueba (referee hostil):**
- Que el ajuste libre se *pueda* llevar a SSEE moviendo $H_0$ es, en parte, **la degeneración misma**: muchos puntos
  de la línea ajustan casi igual. SSEE elige uno *algebraicamente* y resulta estar sobre la línea — eso es
  *consistencia*, no *confirmación*.
- Esto es reponderación del prior de $H_0$, **no** un rerun con $r_d,\Omega_m$ recalculados de forma consistente.

**Lo que sí aporta:** la "distancia SSEE↔CPL" **no es un número absoluto** — depende del prior de $H_0$ con que
compares. Reportar solo el valor al prior de Planck **sub-representa** la consistencia. El siguiente paso riguroso
es **relanzar** CPL con el fondo $H_0$=H_alg (y con MIRA) de forma autoconsistente y medir $(w_0,w_a)$ directamente.

---
### Próximos experimentos (para este mismo laboratorio)
1. Reweight a prior **plano** en $H_0$ (BAO sola, sin prior) → ¿dónde cae la línea sin Planck?  *(ojo al ESS)*
2. **Rerun** real de CPL con prior $H_0$=H_alg autoconsistente (no reweight).
3. Proyectar SSEE sobre el **eje bien-restringido** de la elipse → cuantificar qué predice SSEE en la dirección
   que la BAO *sí* mide (vs la dirección degenerada que no mide).